# Jupyter notebooks

If you've not used Jupyter notebooks before, here are some tips.

A notebook is made of of several cells. Each cell is either made up of text (like this one) or code.

You can execute the code in a cell by pressing ctrl+Enter (cmd+Enter on a mac). Whatever you run will be "remembered" by the Python kernel. So if you assign `a=3` in one cell, you can use `a` later on. Some people (me!) hate Jupyter because your results can depend on the order that you execute cells - this means it's easy to do stupid stuff. You are warned!


In [1]:
import spikeinterface.full as si

# 1. Load a recording

We will use a recording from DANDI.

DANDI (https://dandiarchive.org/) is a _huge_ repository, with a lot of ephys data on it. We'll look at a dataset from Adrian: https://dandiarchive.org/dandiset/000939/

That dataset has a lot of subjects and sessions in it. We'll pick this session: https://dandiarchive.org/dandiset/000939/0.260512.1701/files?location=sub-A3702&page=1

You can either download the 26GB file to your laptop. Or stream parts of it directly from DANDI. Go to Section 1.1 if you want to steam it. Go to Section 1.2 if you've downloaded the file locally and want to load it.

## 1.1 Stream the recording

To load it, we need to find where the data is actually stored. We can find this out by investigating the metdata of the session: https://api.dandiarchive.org/api/dandisets/000939/versions/0.260512.1701/assets/6fe925dd-ea41-4a84-9cee-ca6bb039edcb/

Alternatively, you could use the [dandi python package](https://pynwb.readthedocs.io/en/stable/tutorials/advanced_io/streaming.html) , or [loadi](https://github.com/chrishalcrow/loadi) (which I made). Both help you to load things without having to go get these annoying urls.

In [ ]:
# stream the recording - this will take ~1min, as it's loading some data about the recording
recording = si.read_nwb_recording(
    file_path = 'https://dandiarchive.s3.amazonaws.com/blobs/a8f/800/a8f8003e-4483-4b50-8a45-91ac5971f5d5',
    stream_mode = 'fsspec',
    electrical_series_path = 'acquisition/ElectricalSeries',
)

In [ ]:
recording

## 1.2 Download, then load the recording locally

This is more similar to what you'd do with your own data. Just point the file_path at your download location

In [ ]:
recording = si.read_nwb_recording(
    # EDIT THIS
    file_path = 'path/to/sub-A3702_ses-191126_behavior+ecephys.nwb',
    electrical_series_path = 'acquisition/ElectricalSeries',
)

## 1.3 Exercise: load your own data

You can load lots of different types of data into SpikeInterface. Here's a list: https://spikeinterface.readthedocs.io/en/stable/modules/extractors.html#raw-data-formats

In my lab, we use openephys. So we use `si.read_openephys('path/to/recording')`.

Try to load some of your own data

In [ ]:
my_own_data = si.read_...



# 2. Play with the recording

We now have the `recording` object. We can extract useful properties from it, like the channel locations, sampling frequency...

In [ ]:
print(f"The samping frequency is {recording.sampling_frequency}")

channel_locations = recording.get_channel_locations()
y_locations = channel_locations[:,1]
max_y_location = max(y_locations)
print(f"The max y location of a channel is {max_y_location}. This is measured in microns.")
print(f"The recording start time is {recording.get_time_info()['t_start']}")

In [ ]:
# You can see what else there is to explore by typing out "recording." and pressing the tab bar:
rec...

And you can slice it up, using time or channels

In [ ]:
# Take the first 5 minutes of the recording, and the first 4 channels
recording_slice = recording.time_slice(start_time=0, end_time=60*5).select_channels(channel_ids=recording.channel_ids[:4])

And visualize the raw trace

In [ ]:
%matplotlib widget
si.plot_traces(recording_slice, backend='ipywidgets')

But don't be scared of plotting a full giant recording. SpikeInterface uses a _lazy_ loading system. It doesn't load the entire 26GB recording into RAM. Instead, it only loads the metadata, then will load the raw traces when you request them

In [ ]:
%matplotlib widget
si.plot_traces(recording, backend='ipywidgets')

Hmmmm... That looks gross. We need to...

# 3. Preprocess

It's quite hard to see spikes in raw ephys data, because global eletrical signal dominate. To see them we need to _at least_ bandpass filter and on a high density probe we do a common reference.

In SpikeInterface we think of preprocessing as a chain of steps. It's easiest just to see this in action:


In [ ]:
preprocessed_recording = si.common_reference(si.bandpass_filter(si.depth_order(recording)))

In [ ]:
# let's take a look...
si.plot_traces(preprocessed_recording, backend='ipywidgets', mode = 'map', )

# 3.1 Exercise: Try out some other preprocessing steps

Here's a list: https://spikeinterface.readthedocs.io/en/latest/api.html#api-preprocessing

Maybe try out a normalize, or a whiten, and see if you can still find spikes in the recording.

# 3.2 Exercise: Find some bad channels

From the visualisation, it seems like some channels are bad - probably the electrode on the device is dead. Try out the `si.detect_bad_channels` function to see if you can find them

# 3.3 Exercise: Write a preprocessing pipeline

Once you like your steps, try writing a preprocessing pipleine, as it detailed here: https://spikeinterface.readthedocs.io/en/latest/how_to/build_pipeline_with_dicts.html

## 3.4 Motion correction (optional)

Especially in acute recordings, probe motion is a big problem for spike sorting.

At the moment the most common approach to deal with motion is to 1) estimate the motion 2) try to correct the motion by re-sampling and interpolating the raw recording. There's mounting evidence that we're ok at estimating motion but we're bad at interpolating.

To estimate the motion we can use the `si.compute_motion` function, and then use `si.plot_motion` to visualise the result.

If you want to interpolate, you can then use `si.interpolate

We'll do this below for a 5 minute slice of the recording use the fastest method, called `rigid_fast`. Read more about motion correction here: https://spikeinterface.readthedocs.io/en/latest/how_to/handle_drift.html



In [ ]:
# Note: this will take 5-10mins to run...
motion, motion_info = si.compute_motion(preprocessed_recording.time_slice(0, 5*60), preset='nonrigid_fast_and_accurate', output_motion_info=True)

In [ ]:
si.plot_motion(motion)

In this example, for the first 5 mins and for the simplest motion computation, the motion is _much less_ than the spacing between two electrodes. Hence, I would recommend not doing any motion correction. For real data you should check the entire recording. I recommend the DREDGE preset.

# 4. Sort